# 02 Terrain — Elevation & Slope

Run independently for the seven confirmed parcels. This notebook has no FEMA dependency. It saves terrain statistics, slope classes, elevation/slope maps, and a two-band 30 m UTM raster for integration. Slope is derived at the DEM's native grid before export to the common integration grid. Flooding is handled in `02b_Flooding.ipynb`.

Source: [USGS 3DEP 10 m collection](https://developers.google.com/earth-engine/datasets/catalog/USGS_3DEP_10m_collection). The 30 m integration export is a coarser analysis product, not the native DEM resolution.

Includes a separate surroundings section: elevation/slope statistics and slope-class areas for the full one-mile buffer (including the parcels), plus the surrounding ring (excluding the parcels). New buffer figures use a 10 m UTM export; the existing parcel-only 30 m integration raster is retained.

Map display uses parcel elevation percentiles (2nd–98th) and slope classes (0–2°, 2–5°, 5–10°, ≥10°). Both parcel and buffer maps share the scales. Extreme elevations are shown with end colors and extended colorbar ends; underlying raster values are retained.


Run cells in order. Settings, helper functions, calculations, maps, and exports are separated. Figures are PNG only. Install dependencies once if needed: `earthengine-api geopandas matplotlib numpy pandas shapely pyogrio` plus `geemap rasterio`.

## 1. Imports

Only libraries used by this analysis are loaded.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from shapely import force_2d
import geemap
import numpy as np
import rasterio
from matplotlib.colors import BoundaryNorm, ListedColormap, Normalize
from rasterio.features import geometry_mask
from rasterio.plot import show


## 2. Settings and local paths

Keep data sources, parcel IDs, and output locations in one place.

In [ ]:
EE_PROJECT = "stalwart-glider-279103"
METRIC_CRS = "EPSG:26914"
M_PER_MILE = 1609.344
ACRE_M2 = 4046.8564224
GOAT_ISLAND_PARCEL_IDS = [
    "65079661510030000",
    "65130905010020000",
    "65130905010030000",
    "65130905010050000",
    "65130905010060000",
    "65146334010100000",
    "65146334010140000",
]
PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "Park_Parcels/Park_Parcles.shp").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside goat-island-analysis.")
PARCEL_PATH = PROJECT_ROOT / "Park_Parcels/Park_Parcles.shp"
OUTPUT_DIR = PROJECT_ROOT / "goat_island_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEM_SOURCE = "USGS/3DEP/10m_collection"
TERRAIN_DIR = OUTPUT_DIR / "terrain"
TERRAIN_DIR.mkdir(exist_ok=True)


## 3. Earth Engine connection

Reuse saved authentication when available.

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready. Outputs:", OUTPUT_DIR)


## 4. Seven-parcel boundary

Validate parcel selection and geometry, then create the one-mile study area.

In [ ]:
parcels = gpd.read_file(PARCEL_PATH)
if parcels.crs is None:
    raise ValueError("Park parcels must have a defined coordinate system.")
parcel_numbers = parcels["Parcel_Num"].astype(str).str.strip()
missing_ids = set(GOAT_ISLAND_PARCEL_IDS) - set(parcel_numbers)
if missing_ids:
    raise ValueError(f"Required Goat Island parcels are missing: {sorted(missing_ids)}")
selected = parcels.loc[parcel_numbers.isin(GOAT_ISLAND_PARCEL_IDS)].copy()
if selected.empty:
    raise ValueError("No matching Goat Island project parcels.")
if selected.geometry.isna().any() or selected.geometry.is_empty.any():
    raise ValueError("Selected Goat Island parcels contain missing or empty geometry.")
selected["geometry"] = selected.geometry.map(force_2d).make_valid()
if not selected.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("Selected parcels must contain polygon geometry.")
preserve_gdf = selected.dissolve().reset_index(drop=True).to_crs("EPSG:4326")
BOUNDARY_SOURCE = (
    "Park_Parcels/Park_Parcles.shp; seven project-confirmed Parcel_Num values"
)
preserve_m = preserve_gdf.to_crs(METRIC_CRS)
study1_m = gpd.GeoDataFrame(geometry=preserve_m.buffer(M_PER_MILE), crs=METRIC_CRS)
preserve_area_acres = preserve_m.area.sum() / ACRE_M2
print(
    f"BOUNDARY: {BOUNDARY_SOURCE}; {len(selected)} parcel(s); area = {preserve_area_acres:.1f} acres"
)
preserve_gdf.to_file(OUTPUT_DIR / "preserve_boundary.geojson", driver="GeoJSON")
study1_m.to_crs(4326).to_file(
    OUTPUT_DIR / "one_mile_study_area.geojson", driver="GeoJSON"
)
roi = ee.Geometry(
    json.loads(json.dumps(preserve_gdf.geometry.iloc[0].__geo_interface__))
)
roi1 = ee.Geometry(
    json.loads(json.dumps(study1_m.to_crs(4326).geometry.iloc[0].__geo_interface__))
)


In [ ]:
BOUNDARY_HASH = hashlib.sha256(preserve_gdf.geometry.iloc[0].wkb).hexdigest()
PARCEL_IDS = sorted(GOAT_ISLAND_PARCEL_IDS)


## 5. Reusable helpers

Formatting and export helpers are defined once. Analytical rules are unchanged.

In [ ]:
def save_plot(fig, directory, name):
    """Save a PNG figure and display it in the notebook."""
    fig.savefig(directory / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()


def parcel_display_scales():
    """Derive display limits from valid parcel pixels without changing raster values."""
    reference_path = TERRAIN_DIR / "terrain_one_mile_10m.tif"
    if not reference_path.exists():
        print(
            "Run the one-mile raster export cell below, then rerun this map cell for 10 m parcel-based scales."
        )
        reference_path = TERRAIN_DIR / "terrain_30m.tif"
    with rasterio.open(reference_path) as src:
        parcel_geometry = preserve_m.to_crs(src.crs).geometry.iloc[0]
        inside = geometry_mask(
            [parcel_geometry.__geo_interface__],
            out_shape=src.shape,
            transform=src.transform,
            invert=True,
        )
        elevation = src.read(1, masked=True)
        values = elevation.data[
            inside & ~np.ma.getmaskarray(elevation) & np.isfinite(elevation.data)
        ]
        if values.size == 0:
            raise ValueError("No valid parcel elevation values for display scaling.")
        low, high = np.percentile(values, [2, 98])
        if high <= low:
            high = low + 1
        resolution = abs(src.transform.a)
    scales = {
        "reference": "valid pixels inside seven selected parcels",
        "reference_raster": reference_path.name,
        "reference_grid_m": resolution,
        "elevation_percentiles": [2, 98],
        "elevation_min_m": float(low),
        "elevation_max_m": float(high),
        "slope_class_edges_deg": [0, 2, 5, 10, 90],
        "slope_class_labels": ["0–2°", "2–5°", "5–10°", "≥10°"],
        "data_modified": False,
    }
    (TERRAIN_DIR / "map_display_scales.json").write_text(
        json.dumps(scales, indent=2), encoding="utf-8"
    )
    return scales


def terrain_display_map(raster_path, title, filename):
    """Draw elevation and slope with shared parcel-based display scales."""
    scales = parcel_display_scales()
    low, high = (scales["elevation_min_m"], scales["elevation_max_m"])
    elevation_palette = plt.get_cmap("terrain").copy()
    elevation_palette.set_bad("#eeeeee")
    slope_palette = ListedColormap(["#ffffcc", "#fed976", "#fd8d3c", "#bd0026"])
    slope_palette.set_bad("#eeeeee")
    with rasterio.open(raster_path) as src:
        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        for ax, band, label, palette, norm in zip(
            axes,
            [1, 2],
            ["Elevation — parcel-based color range", "Slope — planning classes"],
            [elevation_palette, slope_palette],
            [Normalize(low, high), BoundaryNorm([0, 2, 5, 10, 90.001], 4)],
        ):
            show(
                src.read(band, masked=True),
                transform=src.transform,
                ax=ax,
                cmap=palette,
                norm=norm,
                interpolation="nearest",
            )
            preserve_m.to_crs(src.crs).boundary.plot(
                ax=ax, color="black", linewidth=1.1
            )
            if "one_mile" in raster_path.name:
                study1_m.to_crs(src.crs).boundary.plot(
                    ax=ax, color="gray", linestyle="--", linewidth=1
                )
            if band == 1:
                fig.colorbar(
                    ax.images[0],
                    ax=ax,
                    shrink=0.75,
                    label="Elevation (m)",
                    extend="both",
                )
            else:
                colorbar = fig.colorbar(
                    ax.images[0],
                    ax=ax,
                    shrink=0.75,
                    spacing="uniform",
                    ticks=[1, 3.5, 7.5, 50],
                    label="Slope class (degrees)",
                )
                colorbar.ax.set_yticklabels(scales["slope_class_labels"])
            ax.set(title=label, xlabel="UTM Easting (m)", ylabel="UTM Northing (m)")
            ax.ticklabel_format(style="plain", useOffset=False)
            x0, x1 = ax.get_xlim()
            y0, y1 = ax.get_ylim()
            sx = x0 + 0.07 * (x1 - x0)
            sy = y0 + 0.06 * (y1 - y0)
            ax.plot([sx, sx + 1000], [sy, sy], color="black", linewidth=2)
            ax.text(sx + 500, sy + 0.015 * (y1 - y0), "1 km", ha="center", fontsize=8)
        fig.suptitle(title)
        caption = f"USGS 3DEP • elevation colors: parcel 2nd–98th percentiles ({low:.1f}–{high:.1f} m); end colors include values beyond this range\nScale reference: {resolution_text(scales)} • same scales on parcel and buffer maps • underlying data unchanged"
        fig.text(0.5, 0.01, caption, ha="center", fontsize=8)
        fig.tight_layout(rect=[0, 0.08, 1, 0.94])
        save_plot(fig, TERRAIN_DIR, filename)
    return scales


def resolution_text(scales):
    """Describe the pixel size used to derive the display scale."""
    return f"{scales['reference_grid_m']:g} m pixels within seven parcels"


## 6. Load elevation and derive slope

Slope is derived on the source grid before clipping to the parcels.

In [ ]:
collection = ee.ImageCollection(DEM_SOURCE).filterBounds(roi1).sort("system:index")
if collection.size().getInfo() == 0:
    raise RuntimeError("No 3DEP tiles intersect the study area.")
native_projection = ee.Image(collection.first()).select("elevation").projection()
dem_full = (
    collection.select("elevation").mosaic().setDefaultProjection(native_projection)
)
dem = dem_full.clip(roi)
slope = ee.Terrain.slope(dem_full).clip(roi)
terrain = dem.rename("elevation_m").addBands(slope.rename("slope_deg"))


## 7. Parcel terrain statistics

Export elevation summaries and slope-class areas at the existing 10 m analysis scale.

In [ ]:
stats = terrain.reduceRegion(
    ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
    roi,
    10,
    maxPixels=100000000,
).getInfo()
pd.DataFrame([stats]).to_csv(TERRAIN_DIR / "terrain_summary.csv", index=False)
records = []
for low, high in [(0, 2), (2, 5), (5, 10), (10, 90.001)]:
    area = (
        ee.Image.pixelArea()
        .updateMask(slope.gte(low).And(slope.lt(high)))
        .reduceRegion(ee.Reducer.sum(), roi, 10, maxPixels=100000000)
        .get("area")
        .getInfo()
    )
    records.append(
        {
            "slope_range_deg": f"{low}-{min(high, 90):g}",
            "area_acres": (area or 0) / ACRE_M2,
        }
    )
table = pd.DataFrame(records)
table["pct_valid_slope_area"] = 100 * table.area_acres / table.area_acres.sum()
table.to_csv(TERRAIN_DIR / "slope_classes.csv", index=False)
display(pd.DataFrame([stats]))
display(table)


## 8. Save the parcel integration raster

Retain the fixed 30 m UTM raster and boundary manifest used by integration.

In [ ]:
xmin, ymin, xmax, ymax = preserve_m.total_bounds
transform = [
    30,
    0,
    float(np.floor(xmin / 30) * 30),
    0,
    -30,
    float(np.ceil(ymax / 30) * 30),
]
path = TERRAIN_DIR / "terrain_30m.tif"
if path.exists():
    path.unlink()
geemap.ee_export_image(
    terrain.unmask(-9999),
    filename=str(path),
    region=roi.bounds(),
    crs=METRIC_CRS,
    crs_transform=transform,
    file_per_band=False,
    timeout=180,
)
if not path.exists():
    raise RuntimeError("Terrain raster download failed.")
with rasterio.open(path, "r+") as src:
    inside = geometry_mask(
        list(
            (
                preserve_m.geometry.__geo_interface__["features"][i]["geometry"]
                for i in range(len(preserve_m))
            )
        ),
        out_shape=(src.height, src.width),
        transform=src.transform,
        invert=True,
    )
    data = src.read()
    data[:, ~inside] = -9999
    src.write(data)
    src.nodata = -9999
    src.set_band_description(1, "elevation_m")
    src.set_band_description(2, "slope_deg")
metadata = {
    "boundary_hash": BOUNDARY_HASH,
    "parcel_ids": PARCEL_IDS,
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "source": DEM_SOURCE,
    "source_image_ids": collection.aggregate_array("system:index").getInfo(),
    "integration_grid_crs": METRIC_CRS,
    "integration_grid_transform": transform,
    "statistics_scale_m": 10,
    "raster": "terrain_30m.tif",
    "integration_resolution_m": 30,
}
(TERRAIN_DIR / "manifest.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)


## 9. Plot parcel slope distribution

This chart uses the parcel statistics; the maps are generated after the surroundings raster exists.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(table.slope_range_deg, table.area_acres, color="steelblue")
ax.set(
    xlabel="Slope range (degrees)",
    ylabel="Area (acres)",
    title="Goat Island slope distribution — 10 m analysis",
)
fig.tight_layout()
save_plot(fig, TERRAIN_DIR, "slope_distribution")


## One-mile terrain surroundings
The **full one-mile buffer** includes all seven selected parcels. The **surrounding ring** excludes them. The two summaries are labeled separately to avoid double-counting. Elevation and slope are calculated from the same 3DEP mosaic as the parcel results; slope is derived before clipping. Statistics use a fixed 10 m UTM grid. Coverage compares valid raster area with projected vector area and may differ slightly along boundaries.

## 10. Define buffer and ring

The full buffer includes the parcels. The ring excludes them. Both use one fixed 10 m UTM grid.

In [ ]:
terrain_buffer = (
    dem_full.rename("elevation_m")
    .addBands(ee.Terrain.slope(dem_full).rename("slope_deg"))
    .clip(roi1)
)
buffer_geometry = study1_m.geometry.iloc[0]
ring_geometry = buffer_geometry.difference(preserve_m.geometry.iloc[0])
ring_wgs = gpd.GeoSeries([ring_geometry], crs=METRIC_CRS).to_crs(4326).iloc[0]
ring_roi = ee.Geometry(json.loads(json.dumps(ring_wgs.__geo_interface__)))
zones = [
    ("Full one-mile buffer (includes parcels)", roi1, buffer_geometry),
    ("Surrounding one-mile ring (excludes parcels)", ring_roi, ring_geometry),
]
buffer_transform = [
    10,
    0,
    float(np.floor(study1_m.total_bounds[0] / 10) * 10),
    0,
    -10,
    float(np.ceil(study1_m.total_bounds[3] / 10) * 10),
]
buffer_grid = {
    "crs": METRIC_CRS,
    "crsTransform": buffer_transform,
    "maxPixels": 100000000,
    "tileScale": 4,
}


## 11. Summarize the surroundings

Keep region labels, coverage checks, terrain summaries, and slope-class areas together.

In [ ]:
surrounding_stats = []
surrounding_slopes = []
for label, region, geometry in zones:
    print("Summarizing:", label, flush=True)
    row = terrain_buffer.reduceRegion(
        ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
        region,
        **buffer_grid,
    ).getInfo()
    row.update(zone=label, geometry_acres=geometry.area / ACRE_M2)
    for band in ["elevation_m", "slope_deg"]:
        area = (
            ee.Image.pixelArea()
            .updateMask(terrain_buffer.select(band).mask())
            .rename("area")
            .reduceRegion(ee.Reducer.sum(), region, **buffer_grid)
            .get("area")
            .getInfo()
        )
        row[band + "_valid_acres"] = (area or 0) / ACRE_M2
        row[band + "_coverage_pct"] = (
            100 * row[band + "_valid_acres"] / row["geometry_acres"]
        )
    surrounding_stats.append(row)
    for low, high in [(0, 2), (2, 5), (5, 10), (10, 90.001)]:
        binary = (
            terrain_buffer.select("slope_deg")
            .gte(low)
            .And(terrain_buffer.select("slope_deg").lt(high))
        )
        area = (
            ee.Image.pixelArea()
            .updateMask(binary)
            .rename("area")
            .reduceRegion(ee.Reducer.sum(), region, **buffer_grid)
            .get("area")
            .getInfo()
        )
        acres = (area or 0) / ACRE_M2
        surrounding_slopes.append(
            {
                "zone": label,
                "slope_range_deg": f"{low}-{min(high, 90):g}",
                "area_acres": acres,
                "pct_valid_slope_area": (
                    100 * acres / row["slope_deg_valid_acres"]
                    if row["slope_deg_valid_acres"]
                    else np.nan
                ),
            }
        )
surrounding_stats_df = pd.DataFrame(surrounding_stats)
surrounding_slopes_df = pd.DataFrame(surrounding_slopes)
if (
    surrounding_stats_df.elevation_m_mean.isna().any()
    or surrounding_stats_df.slope_deg_mean.isna().any()
):
    raise RuntimeError("Missing terrain observations in a surroundings zone.")
surrounding_stats_df.to_csv(
    TERRAIN_DIR / "terrain_surroundings_summary.csv", index=False
)
surrounding_slopes_df.to_csv(
    TERRAIN_DIR / "slope_classes_surroundings.csv", index=False
)
display(surrounding_stats_df)
display(surrounding_slopes_df)
zone_gdf = gpd.GeoDataFrame(
    {"zone": [z[0] for z in zones]}, geometry=[z[2] for z in zones], crs=METRIC_CRS
)
zone_gdf.to_crs(4326).to_file(
    TERRAIN_DIR / "terrain_surroundings_zones.geojson", driver="GeoJSON"
)
for row in surrounding_stats:
    if min(row["elevation_m_coverage_pct"], row["slope_deg_coverage_pct"]) < 98:
        print("WARNING: Review incomplete raster coverage for", row["zone"])


## 12. Export the 10 m surroundings raster

Save both bands, retain NoData handling, and add surroundings provenance to the manifest.

In [ ]:
buffer_path = TERRAIN_DIR / "terrain_one_mile_10m.tif"
if buffer_path.exists():
    buffer_path.unlink()
geemap.ee_export_image(
    terrain_buffer.unmask(-9999),
    filename=str(buffer_path),
    region=roi1.bounds(),
    crs=METRIC_CRS,
    crs_transform=buffer_transform,
    file_per_band=False,
    timeout=180,
)
if not buffer_path.exists():
    raise RuntimeError("One-mile terrain raster export failed.")
with rasterio.open(buffer_path, "r+") as src:
    inside = geometry_mask(
        [buffer_geometry.__geo_interface__],
        out_shape=src.shape,
        transform=src.transform,
        invert=True,
    )
    values = src.read()
    values[:, ~inside] = -9999
    src.write(values)
    src.nodata = -9999
    src.set_band_description(1, "elevation_m")
    src.set_band_description(2, "slope_deg")
metadata = json.loads((TERRAIN_DIR / "manifest.json").read_text())
metadata["surroundings"] = {
    "buffer_distance_m": M_PER_MILE,
    "buffer_includes_parcels": True,
    "ring_excludes_parcels": True,
    "statistics_grid_m": 10,
    "raster_grid_m": 10,
    "crs": METRIC_CRS,
    "transform": buffer_transform,
    "raster": "terrain_one_mile_10m.tif",
    "summary": "terrain_surroundings_summary.csv",
    "slope_classes": "slope_classes_surroundings.csv",
}
(TERRAIN_DIR / "manifest.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)


## 13. Draw parcel and buffer maps

Both maps share parcel elevation percentiles and slope classes. Raster values stay unchanged.

In [ ]:
terrain_display_map(
    TERRAIN_DIR / "terrain_30m.tif",
    "Goat Island terrain — seven parcels; display emphasizes parcel variation",
    "terrain_maps",
)
display_scales = terrain_display_map(
    buffer_path,
    "Goat Island one-mile terrain — colors referenced to seven parcels",
    "terrain_one_mile_maps",
)
print(
    "Elevation color range (parcel 2nd–98th percentiles):",
    round(display_scales["elevation_min_m"], 2),
    "to",
    round(display_scales["elevation_max_m"], 2),
    "m",
)


## 14. Compare surrounding slope distributions

Compare the full buffer and surrounding ring using percentages.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (label, _, _) in zip(axes, zones):
    subset = surrounding_slopes_df[surrounding_slopes_df.zone.eq(label)]
    ax.bar(subset.slope_range_deg, subset.pct_valid_slope_area, color="steelblue")
    ax.set(
        title=label.replace(" (", "\n("),
        xlabel="Slope range (degrees)",
        ylabel="Valid slope area (%)",
        ylim=(0, 100),
    )
    for i, row in enumerate(subset.itertuples()):
        ax.text(
            i,
            row.pct_valid_slope_area + 1,
            f"{row.pct_valid_slope_area:.1f}%",
            ha="center",
            fontsize=9,
        )
fig.suptitle("Goat Island surroundings — slope distribution")
fig.tight_layout()
save_plot(fig, TERRAIN_DIR, "slope_distribution_surroundings")
